# Dataset

# Prompts

In [ ]:
import os
import zipfile
from google.colab import files

In [ ]:
def upload_and_unzip():
    """Upload a zip file and unzip it in the current working directory."""
    print("Please upload your ZIP file:")
    uploaded = files.upload()
    zip_filename = list(uploaded.keys())[0]

    # Extract to a folder named after the zip (without extension)
    extract_dir = os.path.splitext(zip_filename)[0]
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Files extracted to: {extract_dir}")
    return extract_dir

In [ ]:
extract_dir = upload_and_unzip()

Please upload your ZIP file:


Saving Prompts.zip to Prompts (8).zip
Files extracted to: Prompts (8)


In [ ]:
def load_text_files(directory):
    """
    Load all .txt files in directory and return a nested dictionary
    data[type1][type2] = file contents
    """
    data = {}

    for root, _, files in os.walk(directory):
        for fname in files:
            if fname.endswith(".txt"):
                parts = fname.replace(".txt", "").split("_")
                # Expect pattern: system_<type1>_<type2>.txt
                if len(parts) >= 3:
                    type1 = parts[1]
                    type2 = parts[2]
                    fpath = os.path.join(root, fname)
                    with open(fpath, "r", encoding="utf-8") as f:
                        content = f.read()
                    if type1 not in data:
                        data[type1] = {}
                    data[type1][type2] = content
    return data

In [ ]:
data = load_text_files(extract_dir)


In [ ]:
for type1, subtypes in data.items():
    print(f"\n=== {type1.upper()} ===")
    for type2, content in subtypes.items():
      word_count = len(content.split())
      print(f"  -> {type2} ({len(content)} chars) ({word_count} words)")



=== STY ===
  -> awkward (767 chars) (119 words)
  -> supervisor (405 chars) (62 words)

=== ACC ===
  -> supervisor (1102 chars) (169 words)
  -> omission (427 chars) (71 words)
  -> mistranslation (757 chars) (119 words)
  -> name (483 chars) (79 words)
  -> untranslated (690 chars) (107 words)
  -> addition (472 chars) (76 words)

=== TERM ===
  -> inappropriate (959 chars) (151 words)
  -> supervisor (794 chars) (124 words)
  -> inconsistent (393 chars) (57 words)

=== LOC ===
  -> time (511 chars) (86 words)
  -> monetary (562 chars) (90 words)
  -> supervisor (921 chars) (146 words)

=== EVALUATION ===
  -> spreplica (334 chars) (41 words)
  -> first (550 chars) (75 words)
  -> selfreflection (472 chars) (62 words)
  -> supervisor (966 chars) (126 words)

=== LINC ===
  -> grammar (606 chars) (88 words)
  -> spelling (468 chars) (69 words)
  -> punctuation (307 chars) (42 words)
  -> character (316 chars) (47 words)
  -> supervisor (730 chars) (108 words)


In [ ]:
import re

def fill_template(text, variables):
    def replacer(match):
        key = match.group(1)  # get variable name inside {}
        return str(variables.get(key, f"{{{key}}}"))  # keep {key} if missing

    # Find {variable_name} patterns
    return re.sub(r"\{(\w+)\}", replacer, text)

In [ ]:
test_vars = {"pt_esp_error":"omissão", "pt_sup_error":"acurácia", "en_source":"The cat is on the table", "pt_machine_translation": "O gato está na mesa",
             "severidade":"none", "raciocinio":"O texto traduz palavra por palavra sem nenhuma omissão", "proposta":""}
changedTemplate = fill_template(data["evaluation"]["supervisor"], test_vars)
print(changedTemplate)


Texto Fonte: The cat is on the table
Tradução de Máquina: O gato está na mesa
Um de seus colegas, especialista no erro de omissão analisou o texto fonte e a tradução e detectou apenas erros de omissão escolhendo erro ∈ {severo,pequeno,none} provendo raciocínio em português e uma proposta de texto corrigido
Avaliação de erro: none
Raciocínio:  O texto traduz palavra por palavra sem nenhuma omissão
Proposta de texto corrigido: 
Como supervisor de acurácia você deve escolher entre: concordar ou discordar da análise do especialista, se discordar apresentar o raciocínio, sua nova escolha de erro para a tradução (entre {severo,pequeno,none} ), com sua proposta de correção, no seguinte formato json:
{"raciocinio":"raciocinio para escolher o erro a seguir", “concordancia”:”concordo| discordo”,"erro":"severo|pequeno|none","proposta":"uma tradução do texto fonte onde o problema foi corrigido"}
Se erro = none ou concordancia = “concordo”, “proposta” será uma string vazia ""


In [ ]:
metas = {}
metas["acc"] = {}
metas["acc"]["pt_sup_error"] = "acurácia"
metas["acc"]["addition"] = "adição"
metas["acc"]["mistranslation"] = "tradução incorreta"
metas["acc"]["name"] = "consistência de nomes"
metas["acc"]["untranslated"] = "não tradução"
metas["acc"]["omission"] = "omissão"


metas["linc"] = {}
metas["linc"]["pt_sup_error"] = "convenção linguística"
metas["linc"]["grammar"] = "gramática"
metas["linc"]["punctuation"] = "pontuação"
metas["linc"]["spelling"] = "ortografia incorreta"
metas["linc"]["character"] = "codificação de caracteres"

metas["loc"] = {}
metas["loc"]["pt_sup_error"] = "convenção do país"
metas["loc"]["time"] = "formato de hora"
metas["loc"]["monetary"] = "convenção monetária"

metas["sty"] = {}
metas["sty"]["pt_sup_error"] = "estilo incompatível"
metas["sty"]["awkward"] = "texto não natural"

metas["term"] = {}
metas["term"]["pt_sup_error"] = "terminologia"
metas["term"]["inappropriate"] = "terminologia inapropriada"
metas["term"]["inconsistent"] = "inconsistência de terminologia"


# Agents

In [ ]:
from openai import OpenAI
import json

KEY = ''
MODEL_OPENAI = 'gpt-5-nano'
REASONING = 'minimal'
VERBOSITY = 'low'
os.environ["OPENAI_API_KEY"] = KEY
client = OpenAI()

def callGPT(history):
  response = client.responses.create(
      model=MODEL_OPENAI,
      input=history,
      text={
          "format": {
          "type": "json_object"
          },
          "verbosity": VERBOSITY
      },
      reasoning={
          "effort": REASONING
      },
      tools=[],
      store=False,
      include=[
      ]
  )
  raw_output = response.output[1].content[0].text

  parsed_output = json.loads(raw_output)
  #parsed_output['reasoning'] = ''
  #print(type(parsed_output))  # <class 'dict'>
  #print(parsed_output)

  return parsed_output


In [ ]:
def appendHistory(history,role,prompt):
  field = "input_text"
  if (role=="assistant"):
    field = "output_text"
  content = {
        "role": role,
        "content": [
            {
            "type": field,
            "text": prompt
            }
        ]
  }
  history.append(content)
  return history

In [ ]:
def singleEvaluation(source,machineTranslation,type1,type2, tries = 0, printInfo=True):
  if (tries>=3):
    return {"erro":"none","proposta":"","raciocinio":"LLM incapaz de trazer uma resposta valida"}

  sp_vars = {"pt_esp_error":metas[type1][type2], "pt_sup_error":metas[type1]["pt_sup_error"], "en_source":source, "pt_machine_translation": machineTranslation}
  sphistory = []

  specialistEvaluation = fill_template(data["evaluation"]["first"], sp_vars)

  if (printInfo):
    print(data[type1][type2])
    print("SP:",specialistEvaluation)

  sphistory = appendHistory(sphistory,"developer",data[type1][type2])
  sphistory = appendHistory(sphistory,"user",specialistEvaluation)
  #print(sphistory)

  specialistResponse = callGPT(sphistory)

  if (printInfo):
    print("Severidade:",specialistResponse.get("erro"))
    print("Raciocinio:",specialistResponse.get("raciocinio"))
    print("Proposta:",specialistResponse.get("proposta"))


  #if (specialistResponse.get("erro")=="none"):
  #  return specialistResponse
  if (specialistResponse.get("erro")!="severo" and specialistResponse.get("erro")!="pequeno" and specialistResponse.get("erro")!="none"):
    print("Unknown Output From LLM:", specialistResponse)
    return singleEvaluation(source,machineTranslation,type1,type2, tries+1)


  #Check if we need the supervisor:
  sp_vars["severidade"] = specialistResponse.get("erro")
  sp_vars["raciocinio"] = specialistResponse.get("raciocinio") or specialistResponse.get("raciocínio")
  sp_vars["proposta"] = specialistResponse.get("proposta")
  selfreflection =  fill_template(data["evaluation"]["selfreflection"], sp_vars)
  reflectionHistory=[]
  reflectionHistory = appendHistory(reflectionHistory,"developer",data[type1][type2])
  reflectionHistory = appendHistory(reflectionHistory,"user",selfreflection)
  #print (selfreflection)

  selfReflectionResponse = callGPT(reflectionHistory)
  confianca = selfReflectionResponse.get("confianca") or selfReflectionResponse.get("confiança")
  if (printInfo):
    print("SelfReflection:",selfReflectionResponse)
  if (confianca=="alta"):
    return specialistResponse

  #If we need the supervisor
  supervisor = fill_template(data["evaluation"]["supervisor"], sp_vars)
  supervisorHistory=[]
  supervisorHistory = appendHistory(supervisorHistory,"developer",data[type1]["supervisor"])
  supervisorHistory = appendHistory(supervisorHistory,"user",supervisor)
  print("SupervisorHistory:",supervisorHistory)
  supervisorResponse = callGPT(supervisorHistory)

  print("SupervisorOutput:",supervisorResponse)
  if (supervisorResponse.get("concordancia")=="concordo" and supervisorResponse.get("erro")==specialistResponse.get("erro")):
    return specialistResponse

  #if supervisor disagrees, try again
  sp_vars["severidade_su"] = supervisorResponse.get("erro")
  sp_vars["raciocinio_su"] = supervisorResponse.get("raciocinio") or supervisorResponse.get("raciocínio")
  sp_vars["proposta_su"] = supervisorResponse.get("proposta")
  replica =  fill_template(data["evaluation"]["spreplica"], sp_vars)

  specialistDebateHistory = []
  specialistDebateHistory = appendHistory(specialistDebateHistory,"developer",data[type1][type2])
  specialistDebateHistory = appendHistory(specialistDebateHistory,"user",specialistEvaluation)
  specialistDebateHistory = appendHistory(specialistDebateHistory,"assistant",replica)

  prompt = 'Você concorda com a colocação de seu supervisor ou discorda? Se discordar, melhore seu raciocinio, proponha novamente um nivel de erro e proposta de tradução de en para pt-br em json com o formato: {"raciocinio":"seu raciocinio para a resposta",“concordancia”:”concordo|discordo”, "erro":"severo|pequeno|none", "proposta":"proposta de correção da tradução de máquina"}'
  specialistDebateHistory = appendHistory(specialistDebateHistory,"user",prompt)
  print("FinalHistory:",specialistDebateHistory)
  specialistFinalResponse = callGPT(specialistDebateHistory)
  print("Final:",specialistFinalResponse)
  return specialistFinalResponse

  #specialist_1 =
  #sup_vars = {"pt_esp_error":metas[type1][type2], "pt_sup_error":metas[type1]["pt_sup_error"], "en_source":source, "pt_machine_translation": machineTranslation,
  #           "severidade":"none", "raciocinio":"O texto traduz palavra por palavra sem nenhuma omissão", "proposta":""}
  #supervisorEvaluation = fill_template(data["evaluation"]["supervisor"], sup_vars)



In [ ]:
singleEvaluation("If a man will begin with certainties, he shall end in doubts; but if he will be content to begin with doubts he shall end in certainties.","Se um homem começar com certezas, ele terminará em dúvidas; mas se ele estiver contente em começar com dúvidas, ele terminará em certezas.","acc","addition")

You are a Translation Quality Evaluation Agent. Detect only addition errors: error occuring in the target translation that includes content not present in the source text.  Ou seja, termos adicionados durante a tradução, que não correspondem a termos do texto original.
Exemplos: a translation includes additional words and meaning, that do not exist in the original source: “I went to the shopping with my sister” becomes “Eu fui ao shopping com minha irmã e suas amigas”
SP: Analise o texto fonte e a tradução e detecte apenas erros de adição, escolha erro ∈ {severo,pequeno,none}. Provenha raciocínio (reasoning) em português e se erro != none, uma proposta de texto corrigido, Output: somente um objeto JSON com as chaves:
{"raciocinio":"raciocinio para escolher o erro a seguir","erro":"severo|pequeno|none","proposta":"uma tradução do texto fonte onde o problema de adição foi corrigido"}
Se erro = none, “proposta” será uma string vazia ""

Texto Fonte: If a man will begin with certainties, h

{'raciocinio': 'A tradução apresenta uma diferença de sentido, mas não vejo termos adicionados que não existam no texto original. A estrutura e o conteúdo refletem o mesmo enunciado: início com certezas vs início com dúvidas e fim em dúvidas vs fim em certezas.',
 'erro': 'none',
 'proposta': ''}

In [ ]:
import threading
results = []


def callAllAgents(source,translation,max_threads=4):
  semaphore = threading.Semaphore(max_threads)
  threads = []
  results = {}
  def worker(t1, t2):
        with semaphore:  # limit number of active threads
            try:
                result = singleEvaluation(source, translation, t1, t2, printInfo=False)
                results[f"{t1}/{t2}"] = result
            except Exception as e:
                print(f"[ERROR] {t1}/{t2} failed: {e}")

  for type1, subtypes in data.items():
        if type1 == "evaluation":
            continue  # skip this entire group

        print(f"\n=== {type1.upper()} ===")

        for type2, content in subtypes.items():
            if type2 == "supervisor":
                continue  # skip this agent type

            word_count = len(content.split())
            print(f"  -> {type2} ({len(content)} chars) ({word_count} words)")

            # Launch each evaluation in its own thread
            t = threading.Thread(target=worker, args=(type1, type2))
            t.start()
            threads.append(t)

  # Wait for all threads to complete
  for t in threads:
      t.join()
  return results



In [ ]:
''' #Single Thread For Debugging
def callAllAgents(source, translation):
    results = {}

    for type1, subtypes in data.items():
        if type1 == "evaluation":
            continue  # skip this entire group

        print(f"\n=== {type1.upper()} ===")

        for type2, content in subtypes.items():
            if type2 == "supervisor":
                continue  # skip this agent type

            word_count = len(content.split())
            print(f"  -> {type2} ({len(content)} chars) ({word_count} words)")

            # Run directly (no threads)
            result = singleEvaluation(source, translation, type1, type2, printInfo=False)
            results[f"{type1}/{type2}"] = result


    print("\n✅ All evaluations finished (single-threaded).")
    return results
'''

In [ ]:
results = callAllAgents("If a man will begin with certainties, he shall end in doubts; but if he will be content to begin with doubts he shall end in certainties.","Se um homem começar com certezas, ele terminará em dúvidas; mas se ele estiver contente em começar com dúvidas, ele terminará em certezas.")


=== STY ===
  -> awkward (767 chars) (119 words)

=== ACC ===
  -> omission (427 chars) (71 words)
  -> mistranslation (757 chars) (119 words)
  -> name (483 chars) (79 words)
  -> untranslated (690 chars) (107 words)
  -> addition (472 chars) (76 words)

=== TERM ===
  -> inappropriate (959 chars) (151 words)
  -> inconsistent (393 chars) (57 words)

=== LOC ===
  -> time (511 chars) (86 words)
  -> monetary (562 chars) (90 words)

=== LINC ===
  -> grammar (606 chars) (88 words)
  -> spelling (468 chars) (69 words)
  -> punctuation (307 chars) (42 words)
  -> character (316 chars) (47 words)
SupervisorHistory: [{'role': 'developer', 'content': [{'type': 'input_text', 'text': 'You are a Translation Quality Evaluation Supervisor Agent. Detect only accuracy errors: errors occurring when the target content does not accurately correspond to the propositional content of the source text because of distortion, omission, or addition to the message.  Ou seja, erros que alteram o significado d

In [ ]:
from pprint import pprint
pprint(results)

{'acc/addition': {'erro': 'pequeno',
                  'proposta': 'Se um homem começar com certezas, ele terminará '
                              'em dúvidas; mas se estiver contente em começar '
                              'com dúvidas, ele terminará em certezas.',
                  'raciocinio': 'A tradução apresenta conteúdo adicional não '
                                'presente no texto fonte em português: a '
                                "expressão final 'ele terminará em certezas' "
                                "repete a ideia de conclusão com 'certezas', "
                                'mas não há no original um ajuste explícito '
                                'que justifique essa conclusão oposta após '
                                "'doubts'. O parágrafo original sugere que "
                                'começar com dúvidas leva a certezas, o '
                                'inverso é apresentado na conclusão da '
                                'tradu

In [ ]:
def computeScore(evalResults):
    grade = 100

    for key, result in evalResults.items():
        # ensure result is a dict and has the key 'erro'
        if not isinstance(result, dict):
            continue

        erro = result.get("erro")
        if erro == "pequeno":
            grade -= 1
        elif erro == "severo":
            grade -= 5

    # keep grade within 0–100 range
    grade = max(0, min(100, grade))
    return grade


In [ ]:
computeScore(results)

96

#Métricas

# Avaliações